# CHAOSS DPG Metric Demo

Prototype scoring notebook for the 10-row DPG Registry pilot. Placeholder predictions demonstrate metric behavior and should be replaced with real classifier outputs after backend runtime setup.

In [ ]:
import pandas as pd

def parse_sdgs(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    return {part.strip().upper() for part in str(value).split(';') if part.strip()}

df = pd.read_csv('../chaoss-dpg-10-row-pilot.csv')
df['registry_set'] = df['registry_sdgs'].apply(parse_sdgs)
df['predicted_set'] = df['predicted_sdgs'].apply(parse_sdgs)
df.head()

In [ ]:
def overlap(row):
    union = row.registry_set | row.predicted_set
    if not union:
        return 0.0
    return len(row.registry_set & row.predicted_set) / len(union)

df['computed_exact_match'] = df['registry_set'] == df['predicted_set']
df['computed_overlap_score'] = df.apply(overlap, axis=1)
df['computed_top_k_hit'] = df.apply(lambda row: bool(row.registry_set & row.predicted_set), axis=1)
df['false_positives'] = df.apply(lambda row: '; '.join(sorted(row.predicted_set - row.registry_set)), axis=1)
df['false_negatives'] = df.apply(lambda row: '; '.join(sorted(row.registry_set - row.predicted_set)), axis=1)

df[['project_name', 'computed_exact_match', 'computed_overlap_score', 'computed_top_k_hit', 'false_positives', 'false_negatives']]

In [ ]:
summary = {
    'rows': len(df),
    'exact_match_rate': float(df['computed_exact_match'].mean()),
    'mean_overlap_score': float(df['computed_overlap_score'].mean()),
    'top_k_hit_rate': float(df['computed_top_k_hit'].mean()),
}
summary